# Session 1 — Your turn: Reading a dataset

A new table has arrived: 300 delivery orders from three countries. Before anything is modelled, read it the way the session taught: what is a row, what is missing,
what is impossible, what is duplicated, and how much can a rate be trusted?

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd

def make_orders(seed=7, n=300):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"order_id": [f"O{i:04d}" for i in range(n)], "items": rng.integers(1, 8, n),
                       "price": np.round(rng.gamma(4, 12, n), 2),
                       "country": rng.choice(["DZ", "FR", "ES"], n, p=[0.55, 0.30, 0.15]),
                       "late": rng.choice([0, 1], n, p=[0.8, 0.2])})
    df.loc[rng.choice(n, 12, replace=False), "price"] = np.nan          # planted: missing prices
    df.loc[rng.choice(n, 9, replace=False), "country"] = np.nan         # planted: missing countries
    df.loc[rng.choice(n, 3, replace=False), "items"] = -1               # planted: impossible values
    return pd.concat([df, df.iloc[[5, 40]]], ignore_index=True)         # planted: two rows that appear twice

df = make_orders()
print(df.shape)
df.head()
from aij_check import check, close, summary

(302, 5)


## Task 1: Count the planted problems

Fill the dictionary `problems` with four numbers:
`duplicates` (rows that repeat an earlier row exactly), `missing_price`, `missing_country`, and `impossible_items` (an order cannot have zero or fewer items).
Then make `clean` by dropping the duplicates, setting the impossible item counts to missing (`np.nan`), and resetting the index.

In [2]:
problems = {"duplicates": int(df.duplicated().sum()), "missing_price": int(df["price"].isna().sum()),
            "missing_country": int(df["country"].isna().sum()), "impossible_items": int((df["items"] <= 0).sum())}
clean = df.drop_duplicates().reset_index(drop=True)
clean.loc[clean["items"] <= 0, "items"] = np.nan
print(problems)

{'duplicates': 2, 'missing_price': 12, 'missing_country': 9, 'impossible_items': 3}


In [3]:
check("duplicates counted", lambda: problems["duplicates"] == 2)
check("missing prices counted", lambda: problems["missing_price"] == int(df["price"].isna().sum()), "Count the missing values of the 'price' column on the original table.")
check("missing countries counted", lambda: problems["missing_country"] == int(df["country"].isna().sum()))
check("impossible item counts found", lambda: problems["impossible_items"] == 3, "Look for items less than 1.")
check("clean has no repeated row", lambda: not clean.duplicated().any() and len(clean) == len(df) - 2)
check("clean has no impossible item counts", lambda: (clean["items"].dropna() > 0).all() and clean["items"].isna().sum() == 3)
summary()

PASS  duplicates counted
PASS  missing prices counted
PASS  missing countries counted
PASS  impossible item counts found
PASS  clean has no repeated row
PASS  clean has no impossible item counts

6 of 6 checks passed.  Well done.


**Why it works.** `duplicated()` marks the second copy of a repeated row; `isna().sum()` counts the gaps per column; a comparison such as `items <= 0` finds values that cannot be real. Turning impossible values into missing ones keeps the row and tells the later steps that the value is unknown.

## Task 2: How much can a rate be trusted?

Compute, for each country in `clean`, the late-delivery rate `rate` (the mean of `late`), the number of orders `n`, and the half-width of a 95% interval `half = 1.96 * sqrt(rate * (1 - rate) / n)`.
Put them in a table `by_country` indexed by country with columns `rate`, `n` and `half`. Then set `widest` to the name of the country whose rate is the least certain.

In [4]:
grouped = clean.groupby("country")["late"].agg(rate="mean", n="count")
grouped["half"] = 1.96 * np.sqrt(grouped["rate"] * (1 - grouped["rate"]) / grouped["n"])
by_country = grouped
widest = by_country["half"].idxmax()
by_country

,rate,n,half
country,,,
DZ,0.186047,172,0.058157
ES,0.280000,50,0.124456
FR,0.217391,69,0.097325


In [5]:
check("one row per country", lambda: set(by_country.index) == {"DZ", "FR", "ES"})
check("the rates are shares between 0 and 1", lambda: by_country["rate"].between(0, 1).all())
check("n adds up to the orders that have a country", lambda: by_country["n"].sum() == clean["country"].notna().sum())
check("half is computed with 1.96 * sqrt(p(1-p)/n)", lambda: close(by_country["half"], 1.96 * np.sqrt(by_country["rate"] * (1 - by_country["rate"]) / by_country["n"]), 1e-9))
check("the widest interval belongs to the country with the fewest orders", lambda: widest == by_country["n"].idxmin(), "Smaller groups give wider intervals.")
summary()

PASS  one row per country
PASS  the rates are shares between 0 and 1
PASS  n adds up to the orders that have a country
PASS  half is computed with 1.96 * sqrt(p(1-p)/n)
PASS  the widest interval belongs to the country with the fewest orders

5 of 5 checks passed.  Well done.


**Why it works.** The interval shrinks with the square root of the group size, so the smallest country always has the widest interval unless its rate is near 0 or 1. A rate on a handful of rows is an anecdote.